# unbroadcast-pattern — ex3: add_back0 / add_back1 — wire unbroadcast into a broadcasting binary back fn

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbroadcast-pattern`. Running the final beacon cell reports progress against the `Backprop: Unbroadcast pattern` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbroadcast pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbroadcast-pattern`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbroadcast-pattern"
DD_SUBTOPIC = "Backprop: Unbroadcast pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `add_back0` uses unbroadcast — bridge from unbroadcast to a real back fn

Ex1 wrote `unbroadcast` for leading + size-1 axes; ex2 verified the combined case and idempotence. The deepening move PLUGS unbroadcast into the smallest real binary back fn: addition.

```python
# Forward (with broadcasting): out = x + y  where x.shape != y.shape
# Mathematical gradient:        dL/dx = grad_out, dL/dy = grad_out
# BUT grad_out.shape == out.shape (after broadcast) — it doesn't fit
# back into x or y. So each back fn must UNBROADCAST.

def add_back0(grad_out, out, x, y):
    return unbroadcast(grad_out, x)

def add_back1(grad_out, out, x, y):
    return unbroadcast(grad_out, y)
```

**Why every broadcasting binary op needs this.** `add`, `sub`, `mul`, `div` all broadcast. Their math gradients are simple (`grad_out` for add, `grad_out * y` for mul, etc.) — but those shapes match `out`, not the original inputs. Skipping the unbroadcast step gives gradient tensors of the wrong shape and the next op in the chain crashes.

**Equivalence to torch.autograd.** PyTorch's own AddBackward kernel does exactly this — sums the incoming grad across axes the input was broadcast over. We're rebuilding that pattern from scratch.

**The exemplar isn't just `add`.** Once `add_back0/1` works, the same template applies to every elementwise binary op. The ONLY thing that changes is the math factor (`grad_out`, `grad_out * y`, `grad_out / y`, ...) BEFORE the unbroadcast call.

### Exercise 3 — add_back0 / add_back1 — wire unbroadcast into a broadcasting binary back fn

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the unbroadcast pattern inside `add_back0` and `add_back1` so that even when `x + y` broadcast `x` against `y`, each returned grad has shape matching its respective input — bridging the elementwise math gradient and the input-shape contract.
> Keywords: unbroadcast, add-back, binary-op, broadcasting
> ```

**KCs targeted:** `unbroadcast-pattern`, `binary-back-fn-restores-input-shape`

Implement `ex3_add_back()` returning a dict with three keys: `'unbroadcast'`, `'add_back0'`, `'add_back1'`.

1. `unbroadcast(grad, original)` — given a `grad` whose shape is the broadcasted output shape, sum it back down to `original.shape`:
   - Step A: while `grad.ndim > original.ndim`, `grad = grad.sum(dim=0)`.
   - Step B: for each axis `i` where `original.shape[i] == 1` and `grad.shape[i] != 1`, `grad = grad.sum(dim=i, keepdim=True)`.
   - Return the result. Final shape must equal `original.shape`.

2. `add_back0(grad_out, out, x, y) -> grad_x` — for `out = x + y` (possibly broadcasting). Math gradient is `1`, so locally `grad_x = grad_out`, then unbroadcast to `x.shape`.

3. `add_back1(grad_out, out, x, y) -> grad_y` — symmetric: `grad_y = grad_out`, then unbroadcast to `y.shape`.

Constraint: `add_back0` and `add_back1` MUST call your `unbroadcast` helper — do not inline its body in either of them.

In [ ]:
def ex3_add_back():
    """Return {unbroadcast, add_back0, add_back1} — binary back fns using unbroadcast."""
    raise NotImplementedError()


def _test_ex3():
    result = ex3_add_back()
    unbroadcast = result['unbroadcast']
    add_back0 = result['add_back0']
    add_back1 = result['add_back1']

    # === unbroadcast: leading-axes only ===
    g = t.ones(2, 3, 4)
    orig = t.zeros(3, 4)
    out = unbroadcast(g, orig)
    assert out.shape == orig.shape, f'leading-only: shape {out.shape}'
    assert t.allclose(out, t.full((3, 4), 2.0))

    # === unbroadcast: size-1 only ===
    g = t.ones(3, 5, 4)
    orig = t.zeros(3, 1, 4)
    out = unbroadcast(g, orig)
    assert out.shape == orig.shape
    assert t.allclose(out, t.full((3, 1, 4), 5.0))

    # === unbroadcast: combined leading + size-1 ===
    g = t.ones(2, 3, 1, 4)
    orig = t.zeros(3, 1, 1)
    out = unbroadcast(g, orig)
    assert out.shape == orig.shape, f'combined: got {out.shape}'
    # 2 leading axes (2*1=2) * 4 (broadcast in axis 2 of orig if orig.shape[1]==1)
    # Easier: just verify shape contract.

    # === add_back: no broadcasting (same shape both inputs) ===
    x = t.tensor([[1.0, 2.0], [3.0, 4.0]])
    y = t.tensor([[10.0, 20.0], [30.0, 40.0]])
    out = x + y
    grad_out = t.ones_like(out)
    g0 = add_back0(grad_out, out, x, y)
    g1 = add_back1(grad_out, out, x, y)
    assert g0.shape == x.shape
    assert g1.shape == y.shape
    assert t.allclose(g0, t.ones_like(x))
    assert t.allclose(g1, t.ones_like(y))

    # === add_back: leading-axis broadcast ===
    # x.shape == (4,), y.shape == (3, 4) → out.shape == (3, 4).
    # grad_x must be summed across leading axis 0 to get back to (4,).
    x = t.ones(4)
    y = t.ones(3, 4)
    out = x + y
    assert out.shape == (3, 4)
    grad_out = t.ones_like(out)
    g0 = add_back0(grad_out, out, x, y)
    g1 = add_back1(grad_out, out, x, y)
    assert g0.shape == x.shape, f'g0 shape: {g0.shape}, expected {x.shape}'
    assert g1.shape == y.shape, f'g1 shape: {g1.shape}, expected {y.shape}'
    assert t.allclose(g0, t.full((4,), 3.0))  # summed 3 leading rows
    assert t.allclose(g1, t.ones(3, 4))

    # === add_back: size-1 broadcast ===
    # x.shape == (3, 1), y.shape == (3, 5) → out.shape == (3, 5).
    x = t.ones(3, 1)
    y = t.ones(3, 5)
    out = x + y
    grad_out = t.ones_like(out)
    g0 = add_back0(grad_out, out, x, y)
    g1 = add_back1(grad_out, out, x, y)
    assert g0.shape == (3, 1), f'g0 shape: {g0.shape}'
    assert g1.shape == (3, 5)
    assert t.allclose(g0, t.full((3, 1), 5.0))  # summed 5 cols

    # === Cross-check vs torch.autograd ===
    x = t.randn(4, requires_grad=True)
    y = t.randn(3, 4, requires_grad=True)
    out_t = x + y
    out_t.sum().backward()
    ours_x = add_back0(t.ones_like(out_t), out_t.detach(), x.detach(), y.detach())
    ours_y = add_back1(t.ones_like(out_t), out_t.detach(), x.detach(), y.detach())
    assert t.allclose(x.grad, ours_x), f'autograd x.grad mismatch: {x.grad} vs {ours_x}'
    assert t.allclose(y.grad, ours_y)

    # === Combined broadcast (leading + size-1) — the headline case ===
    x = t.ones(1, 4)
    y = t.ones(2, 3, 4)
    out = x + y
    assert out.shape == (2, 3, 4)
    grad_out = t.ones_like(out)
    g0 = add_back0(grad_out, out, x, y)
    g1 = add_back1(grad_out, out, x, y)
    assert g0.shape == x.shape, f'g0 shape: {g0.shape}, expected {x.shape}'
    assert g1.shape == y.shape
    # After 2 leading-axis peels (2*3=6 rows), shape (1,4). All 6 contributed.
    assert t.allclose(g0, t.full((1, 4), 6.0))
    assert t.allclose(g1, t.ones(2, 3, 4))
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_add_back():
    def unbroadcast(grad, original):
        # Step A: peel leading axes.
        while grad.ndim > original.ndim:
            grad = grad.sum(dim=0)
        # Step B: collapse size-1 axes that got expanded.
        for i, (g_dim, o_dim) in enumerate(zip(grad.shape, original.shape)):
            if o_dim == 1 and g_dim != 1:
                grad = grad.sum(dim=i, keepdim=True)
        return grad

    def add_back0(grad_out, out, x, y):
        # d(x+y)/dx = 1 → local grad is grad_out, unbroadcast to x.shape.
        return unbroadcast(grad_out, x)

    def add_back1(grad_out, out, x, y):
        # d(x+y)/dy = 1 → local grad is grad_out, unbroadcast to y.shape.
        return unbroadcast(grad_out, y)

    return {
        'unbroadcast': unbroadcast,
        'add_back0': add_back0,
        'add_back1': add_back1,
    }
```

**Order of unbroadcast steps matters.** Peel leading axes FIRST (step A), then collapse size-1 axes (step B). The reverse order would mis-align dims: a size-1 axis index in `original.shape` doesn't necessarily match the same index in the larger `grad.shape` until you've peeled the leading dims off.

**`add_back0` and `add_back1` are the same shape.** Both have local gradient `1`, so both return `unbroadcast(grad_out, x_or_y)`. This is the ONLY binary op where the two back fns are functionally identical — `mul_back0` is `unbroadcast(grad_out * y, x)`, `div_back0` is `unbroadcast(grad_out / y, x)`, etc. Add is the easiest case to introduce the pattern.

**Composability with the parents-dispatch drill.** Once `add_back0` / `add_back1` are registered into `BACK_FUNCS` at `(t.add, 0)` and `(t.add, 1)`, the dispatcher from the parents-dispatch drill calls them automatically. The whole pattern composes — no special-casing for broadcasting at the dispatcher level.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()